# Figure styling — the `figstyle` module

SoftMobility ships a small matplotlib styling helper, `softmobility.classes.figstyle`, that applies the paper aesthetic (white background, black box, no grid, Helvetica labels at 11 pt, fixed colour palette, orthographic 3D camera) used in every other tutorial.

This notebook is a tour of its main entry points. It produces four example figures, all exported as **true vector PDF** in `figures/`:

1. **2D**: $y_1 = \sin x$ and $y_2 = \cos x$.
2. **3D helicoidal trajectory** at all three sizes (`full`, `half`, `third`).
3. **Sphere assembly** with body axes — the matplotlib-3D depth-ordering canary.
4. **Label add/remove/displace** demo (2D and 3D).

## Setup

In [ ]:
import numpy as np

from softmobility.classes import figstyle as fs

fs.apply()        # set rcParams to the paper aesthetic

FIGDIR = "figures"

## 1 — 2D demo: $\sin x$ and $\cos x$ (half-column)

In [ ]:
x = np.linspace(0.0, 2 * np.pi, 200)
y1 = np.sin(x)
y2 = np.cos(x)

fig, ax = fs.figure(size="half", aspect=4 / 3)
ax.plot(x, y1, color=fs.COLORS["red"], linewidth=1.5, label="sin(x)")
ax.plot(x, y2, color=fs.COLORS["blue"], linewidth=1.5, label="cos(x)")
ax.set_xlim(0, 2 * np.pi)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="lower left", frameon=False)

fs.save(fig, "fig_2d_sin_cos", figdir=FIGDIR)
fig

## 2 — 3D helicoidal trajectory at full / half / third sizes

A helix $\mathbf{r}(t) = (\cos t,\; \sin t,\; t/(6\pi))$ over $t \in [0, 6\pi]$,
projected onto the three back walls of the bounding cube and rendered
at every named size. Every PDF is fully vector.

In [ ]:
t = np.linspace(0.0, 6 * np.pi, 600)
xs = np.cos(t)
ys = np.sin(t)
zs = t / (6 * np.pi)
bounds = ((-1.1, 1.1), (-1.1, 1.1), (-0.05, 1.05))

for size in ("full", "half", "third"):
    fig, ax = fs.figure_3d(size=size, aspect=1.0)
    fs.add_back_panels(ax, *bounds, color="black", width=1.0)
    fs.add_shadow(ax, xs, ys, zs, plane="xy_low", bounds=bounds)
    fs.add_shadow(ax, xs, ys, zs, plane="xz_low", bounds=bounds)
    fs.add_shadow(ax, xs, ys, zs, plane="yz_low", bounds=bounds)
    ax.plot(xs, ys, zs, color=fs.COLORS["red"], linewidth=1.5, label="helix")
    ax.set_xlim(*bounds[0])
    ax.set_ylim(*bounds[1])
    ax.set_zlim(*bounds[2])
    fs.save(fig, f"helix_{size}", figdir=FIGDIR)

# Show the half-size variant for the inline preview.
fig

## 3 — Sphere assembly with body axes (depth-ordering canary)

matplotlib's 3D backend sorts whole *Artists* by depth, not pixels, so
transparent surfaces and the curves drawn on top of them can stack
incorrectly. This is the cell that makes that limitation visible.

In [ ]:
positions = np.array([
    [0.0, 0.0, 0.0],
    [0.6, 0.0, 0.0],
    [0.0, 0.8, 0.0],
    [0.0, 0.0, 1.0],
])
radii = np.array([0.5, 0.1, 0.3, 0.5])

fig, ax = fs.figure_3d(size="half", aspect=1.0)

axis_length = 1.4
fs.add_body_axes(ax, length=axis_length, color=fs.COLORS["black"])

for centre, radius in zip(positions, radii, strict=True):
    fs.add_sphere(ax, centre, radius)

ax.scatter([0], [0], [0], color=fs.COLORS["black"], marker="x", s=30)

bound = float(max(radii.max(), axis_length * 1.2))
ax.set_xlim(-bound, bound)
ax.set_ylim(-bound, bound)
ax.set_zlim(-bound, bound)

fs.save(fig, "fig_sphere_assembly", figdir=FIGDIR)
fig

## 4 — Adding, removing, displacing labels

Same three helpers as in the plotly version; the position argument's
length picks 2D vs 3D automatically. Labels are tracked by the
matplotlib `gid` so `remove_label`/`displace_label` can find them.

In [ ]:
fig_lbl, ax_lbl = fs.figure(size="half", aspect=4 / 3)
ax_lbl.plot(x, y1, color=fs.COLORS["red"], linewidth=1.5)
ax_lbl.plot(x, y2, color=fs.COLORS["blue"], linewidth=1.5)
ax_lbl.set_xlim(0, 2 * np.pi)
ax_lbl.set_ylim(-1.4, 1.4)
ax_lbl.set_xlabel("x")
ax_lbl.set_ylabel("y")

fs.add_label(ax_lbl, (np.pi / 2, 1.35), "max(sin)",
             color=fs.COLORS["red"], anchor="bottom center",
             offset=(0, 0.05), name="sin_max")
fs.add_label(ax_lbl, (np.pi, -1.35), "min(cos)",
             color=fs.COLORS["blue"], anchor="bottom left",
             offset=(0.05, 0.05), name="cos_max")

n_removed = fs.remove_label(ax_lbl, "cos_max")
n_moved = fs.displace_label(ax_lbl, "sin_max", offset=(0.4, -0.1))
print(f"removed {n_removed}, moved {n_moved}")
fig_lbl

### Same helpers in 3D

In [ ]:
fig3, ax3 = fs.figure_3d(size="half", aspect=1.0)
fs.add_body_axes(ax3, length=1.4, color=fs.COLORS["black"], show_labels=True)
for centre, radius in zip(positions, radii, strict=True):
    fs.add_sphere(ax3, centre, radius)
ax3.scatter([0], [0], [0], color=fs.COLORS["black"], marker="x", s=30)

fs.displace_label(ax3, "axis_label_E1", text="x", offset=(0, 0, -0.05))
fs.remove_label(ax3, "axis_label_E2")
fs.displace_label(ax3, "axis_label_E3", new_position=(0.3, 0.0, 1.2))
fs.add_label(ax3, (0.6, 0, 0.0), "sphere 1",
             color=fs.COLORS["red"], anchor="bottom center",
             offset=(0, 0, -0.2), name="sphere1_caption")

bound = float(max(radii.max(), 1.4 * 1.2))
ax3.set_xlim(-bound, bound)
ax3.set_ylim(-bound, bound)
ax3.set_zlim(-bound, bound)
fig3

## 5 — Verify the output is vector

For each saved PDF, count embedded raster images. A vector-only PDF
reports zero. The plotly demo for the same scene reports one image
per file (the whole 3D scene baked into a PNG).

In [ ]:
from pathlib import Path

try:
    import pypdf
except ImportError:
    print("pypdf not installed (`pip install pypdf`) — skipping vector check")
else:
    print(f"{'file':28s}  {'images':>6s}  {'bytes':>9s}")
    print("-" * 50)
    for pdf in sorted(Path(FIGDIR).glob("*.pdf")):
        page = pypdf.PdfReader(str(pdf)).pages[0]
        n_images = 0
        res = page.get("/Resources", {})
        if "/XObject" in res:
            for v in res["/XObject"].values():
                if v.get_object().get("/Subtype") == "/Image":
                    n_images += 1
        print(f"{pdf.name:28s}  {n_images:>6d}  {pdf.stat().st_size:>9d}")